# **Tugas Praktikum 5: Sequential Pattern`**


Nama : Sandi Suryo Nugroho

NRP : 5054251020

Dataset : Online Retail.xlsx

Pada praktikum ini, mahasiswa menemukan pola pembelian yang berurutan dari waktu ke waktu dalam dataset transaksi menggunakan Sequential Pattern Mining. Pola berurutan diekstraksi dengan algoritma PrefixSpan atau GSP, lalu dievaluasi berdasarkan metrik support dan confidence.

## 1. Setup Library dan Dataset

In [ ]:
import pandas as pd
from prefixspan import PrefixSpan
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('Online Retail.xlsx')

print("Dimensi dataset:", df.shape)
print("\nInformasi Dataset:")
df.info()

display(df.head())

Dimensi dataset: (541909, 8)

Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 2. EDA dan Preprocessing

In [ ]:
rows_before = df.shape[0]
invoices_before = df['InvoiceNo'].nunique()
customers_before = df['CustomerID'].nunique()

# Proses pembersihan data
# 1. Menghapus baris dengan Description kosong
df_clean = df.dropna(subset=['Description'])

# 2. Menghapus baris dengan CustomerID kosong
df_clean = df_clean.dropna(subset=['CustomerID'])

# 3. Menghapus transaksi yang dibatalkan (InvoiceNo diawali 'C')
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# 4. Menghapus Quantity bernilai negatif atau nol
df_clean = df_clean[df_clean['Quantity'] > 0]

# Membersihkan whitespace pada kolom Description
df_clean['Description'] = df_clean['Description'].str.strip()

rows_after = df_clean.shape[0]
invoices_after = df_clean['InvoiceNo'].nunique()
customers_after = df_clean['CustomerID'].nunique()

print(f"Sebelum Pembersihan : {rows_before} baris, {invoices_before} invoice, {customers_before} pelanggan")
print(f"Sesudah Pembersihan : {rows_after} baris, {invoices_after} invoice, {customers_after} pelanggan")

Sebelum Pembersihan : 541909 baris, 25900 invoice, 4372 pelanggan
Sesudah Pembersihan : 397924 baris, 18536 invoice, 4339 pelanggan


In [ ]:
# 1. Mengurutkan data berdasarkan CustomerID dan InvoiceDate secara kronologis
df_sorted = df_clean.sort_values(by=['CustomerID', 'InvoiceDate'])

# 2. Menyusun daftar produk (string) berurutan waktu untuk setiap pelanggan
customer_sequences = df_sorted.groupby('CustomerID')['Description'].apply(lambda x: list(dict.fromkeys(x))).tolist()

print(f"Total sequences (jumlah pelanggan unik): {len(customer_sequences)}")
print("\nContoh sequence pelanggan pertama (5 produk pertama):")
print(customer_sequences[0][:5])

Total sequences (jumlah pelanggan unik): 4339

Contoh sequence pelanggan pertama (5 produk pertama):
['MEDIUM CERAMIC TOP STORAGE JAR']


## 3. Penerapan Algoritma Sequential Pattern

In [9]:

ps = PrefixSpan(customer_sequences)

# Menentukan minimum support: 1% dari total pelanggan
min_supp_ratio = 0.01
min_supp_count = int(min_supp_ratio * len(customer_sequences))

print(f"Minimum support count: {min_supp_count} pelanggan (dari total {len(customer_sequences)})")

# Membatasi pencarian hanya untuk pola dengan minimal 2 item berurutan (panjang sequence >= 2)
ps.minlen = 2

frequent_patterns = ps.frequent(min_supp_count)

# Mengurutkan pola berdasarkan nilai support tertinggi (descending)
frequent_patterns.sort(key=lambda x: x[0], reverse=True)

print(f"Berhasil menemukan {len(frequent_patterns)} pola pembelian berurutan.")

Minimum support count: 43 pelanggan (dari total 4339)
Berhasil menemukan 13983 pola pembelian berurutan.


## 4. Evaluasi

In [10]:
# Fungsi untuk menghitung frekuensi kemunculan antecedent dalam sequence pelanggan
def count_antecedent_support(antecedent, sequences):
    count = 0
    for seq in sequences:
        sub_idx = 0
        for item in seq:
            if item == antecedent[sub_idx]:
                sub_idx += 1
                if sub_idx == len(antecedent):
                    count += 1
                    break
    return count

results = []

for count, pattern in frequent_patterns:
    support_ratio = count / len(customer_sequences)
    
    # Antecedent adalah elemen awal pola (sebelum item terakhir)
    antecedent = pattern[:-1]
    
    # Hitung jumlah pelanggan yang membeli antecedent
    antecedent_count = count_antecedent_support(antecedent, customer_sequences)
    
    # Confidence = Support(Antecedent U Consequent) / Support(Antecedent)
    confidence = count / antecedent_count if antecedent_count > 0 else 0
    
    results.append({
        'Pola (Sequence)': " -> ".join(pattern),
        'Support Count': count,
        'Support (%)': round(support_ratio * 100, 2),
        'Confidence (%)': round(confidence * 100, 2)
    })

# Membuat DataFrame evaluasi dan menampilkannya
df_eval = pd.DataFrame(results)

print("Top 10 Pola Berurutan Berdasarkan Support:")
display(df_eval.head(10))

Top 10 Pola Berurutan Berdasarkan Support:


,Pola (Sequence),Support Count,Support (%),Confidence (%)
0,REGENCY CAKESTAND 3 TIER -> SET OF 3 REGENCY C...,240,5.53,27.24
1,PARTY BUNTING -> SPOTTY BUNTING,214,4.93,30.23
2,REGENCY CAKESTAND 3 TIER -> ROSES REGENCY TEAC...,211,4.86,23.95
3,JUMBO BAG RED RETROSPOT -> JUMBO BAG VINTAGE D...,202,4.66,31.81
4,REGENCY CAKESTAND 3 TIER -> GREEN REGENCY TEAC...,197,4.54,22.36
5,PAPER CHAIN KIT 50'S CHRISTMAS -> PAPER CHAIN ...,195,4.49,31.81
6,JUMBO BAG RED RETROSPOT -> JUMBO BAG 50'S CHRI...,188,4.33,29.61
7,REGENCY CAKESTAND 3 TIER -> PINK REGENCY TEACU...,182,4.19,20.66
8,LUNCH BAG RED RETROSPOT -> LUNCH BAG PINK POLK...,181,4.17,34.02
9,JUMBO BAG RED RETROSPOT -> JUMBO BAG APPLES,178,4.10,28.03


## 5. Analisis

**Total Sequences (Pelanggan Unik)**: 4.339 pelanggan.

**Minimum Support Threshold**: 1% (ditetapkan pada minimal 43 pelanggan).

**Total Pola Berurutan Ditemukan**: 13.983 pola.


### Analisis Top 3 Pola Berurutan Terbaik

1. `REGENCY CAKESTAND 3 TIER` $\rightarrow$ `SET OF 3 REGENCY CAKESTAND`
* **Support**: 5,53% (240 pelanggan)

* **Confidence**: 27,24%

Pembelian produk tempat kue tingkat 3 (*cakestand*) memicu pembeli untuk kembali membeli produk set tempat kue serupa. Pola ini menunjukkan kepuasan tinggi terhadap lini barang bertema *Regency*, di mana pelanggan melengkapi koleksi tempat penyajian makanan mereka.


2. `PARTY BUNTING` $\rightarrow$ `SPOTTY BUNTING`
* **Support**: 4,93% (214 pelanggan)

* **Confidence**: 30,23%

Pelanggan yang membeli bendera dekorasi pesta (*Party Bunting*) memiliki kecenderungan tinggi untuk membeli variasi bendera dekorasi bintik (*Spotty Bunting*) di transaksi berikutnya. Pola ini mengindikasikan adanya kebiasaan pembeli dalam memperluas dekorasi acara dengan beragam motif komplementer.


3. `REGENCY CAKESTAND 3 TIER` $\rightarrow$ `ROSES REGENCY TEACUP`
* **Support**: 4,86% (211 pelanggan)

* **Confidence**: 23,95%

Produk `REGENCY CAKESTAND 3 TIER` berfungsi sebagai *anchor product* (produk pemicu utama). Setelah memiliki tempat penyajian kue, pelanggan memicu transaksi susulan berupa cangkir teh bertema mawar (*Roses Regency Teacup*) untuk melengkapi set acara minum teh mereka.


### Implikasi Bisnis & Rekomendasi

* **Pemasaran Berbasis Waktu (*Cross-Selling*)**

Pola pada nomor 8 (`LUNCH BAG RED RETROSPOT` $\rightarrow$ `LUNCH BAG PINK POLKADOT`) memiliki nilai **Confidence tertinggi (34,02%)**. Toko dapat mengirimkan rekomendasi produk bermotif lain (*Pink Polkadot*) melalui email otomatis beberapa hari setelah pelanggan membeli tas makan warna merah (*Red Retrospot*).


* **Manajemen Stok Produk Pemicu (*Anchor Product*)**

Produk seperti `REGENCY CAKESTAND 3 TIER` dan `JUMBO BAG RED RETROSPOT` muncul berulang kali di posisi *antecedent* (awal transaksi) pada Top 10 pola. Pasokan kedua barang ini harus selalu terjaga untuk mendorong rantai transaksi lanjutan pelanggan.